# Historical Baseline: Multi-Season Player–Gameweek Dataset for Fantasy Premier League Forecasting

---

## Abstract

This notebook constructs the historical baseline dataset that underpins the
predictive modelling stage of the Fantasy Premier League (FPL) forecasting
pipeline. Five seasons of officially-published FPL data (2020-21 through
2024-25) are ingested from a curated public mirror [1], harmonised against
schema drift, joined with fixture metadata to recover ground-truth team
identifiers, and enriched with a small set of leakage-safe predictive
features (rolling form windows, consecutive-absence flags, stable
cross-season player identifiers).

The artefact produced — `data/processed/historical_baseline.csv` — is a
single, dtype-consistent player–gameweek table that serves as the *frozen*
training base for the downstream live-ingestion and modelling notebooks.
Under the pipeline's separation-of-concerns design, no current-season data
and no team-relative aggregations (which require both historical and live
rows in memory) are computed here; those responsibilities belong to
`live_ingestion_and_merge.ipynb`.

## References

[1] A. Vaastav, "Fantasy Premier League Historical Data," GitHub
    repository, 2024. [Online]. Available:
    https://github.com/vaastav/Fantasy-Premier-League

## 1. Introduction and Objectives

### 1.1 Motivation

Fantasy Premier League is a player-selection game in which participants
score points based on the real-world performance of professional
footballers across a 38-gameweek season. Forecasting per-player gameweek
points is a non-trivial supervised learning problem characterised by
non-stationary form, fixture-dependent expected difficulty, and frequent
player turnover between seasons. A defensible model requires a defensible
dataset; the purpose of this notebook is to produce that dataset for the
five most recent fully-completed seasons.

### 1.2 Problem Statement

The raw season-level CSVs published in [1] suffer from three structural
issues that, if left unaddressed, would corrupt any downstream model:

1. **Schema drift across seasons.** Column names, presence, and data
   types vary between seasons — for example, `expected_goals` was added
   in 2021-22 and is absent from earlier files.
2. **Unreliable team identifiers in `players_raw.csv`.** The `team`
   column reflects a player's *current* club at the time the file was
   snapshotted, not their club for each individual gameweek. A player
   transferred mid-season is therefore mis-attributed for half of their
   appearances.
3. **Player identity instability.** Players appear under different name
   spellings across seasons (accents, hyphens, suffixes), and FPL `id`
   values are recycled annually. A stable cross-season identifier must
   be constructed.

### 1.3 Objectives

This notebook addresses the above with four concrete deliverables:

- **O1.** Ingest gameweek-level data for seasons 2020-21 through 2024-25
  into a single in-memory dataframe with a unified schema.
- **O2.** Recover the ground-truth per-gameweek team identifier for every
  player by back-deriving it from the fixture table rather than trusting
  `players_raw.csv`.
- **O3.** Construct a stable cross-season player identifier
  (`Player UUID`) keyed on a normalised name representation.
- **O4.** Enrich the dataset with a minimal set of leakage-safe
  predictive features: lagged rolling form (L3, L5) and a consecutive
  zero-minute absence flag.

### 1.4 Scope and Non-Goals

This notebook **does not** perform any of the following — these are the
responsibilities of `live_ingestion_and_merge.ipynb`:

- Acquisition of the current (in-progress) season via the live FPL API.
- Team-relative aggregations (gameweek contribution share, causal
  cumulative contribution, intra-team contribution rank).
- Active-player filtering or zero-minute row pruning.

This separation is deliberate. The historical baseline produced here is
*frozen* once written; only the downstream live-merge notebook is re-run
on a per-gameweek cadence during the season.

## 2. Methodology

### 2.1 Data Sources

| Source | Description | Used for |
| --- | --- | --- |
| `data/{season}/gws/gw{N}.csv` | Per-gameweek player performance rows | Primary observation table |
| `data/{season}/players_raw.csv` | Player metadata snapshot | Names, positions (`element_type`) |
| `data/{season}/fixtures.csv` | Match schedule with home/away teams and difficulty ratings | Ground-truth team identification |
| `data/{season}/teams.csv` | Team ID → name mapping for the season | Human-readable team labels |
| `master_team_list.csv` | Cross-season team mapping fallback | Used when `teams.csv` is missing for a season |

### 2.2 Schema Harmonisation

A canonical column set is enforced across all seasons. Columns absent
from a particular season's raw file are introduced as missing values
(`NaN`) so that downstream concatenation produces a rectangular table.
Numeric identifier columns (`element`, `team`, `Gameweek`,
`opponent_team`) are coerced to nullable `Int64` to preserve missingness
without forcing implicit `float` casts.

### 2.3 Fixture-Derived Team Identification

The single most important methodological decision in this notebook is
**not** to use `players_raw.team` as the per-gameweek team of a player.
That column reflects the snapshot state of the FPL platform when the
file was published and is incorrect for any player transferred during
the season.

Instead, the per-gameweek team is recovered as follows. For each
gameweek $w$ and fixture $(h, a)$ in `fixtures.csv`, two rows are
emitted:

- $(\text{Gameweek}=w,\ \text{OppKey}=a) \Rightarrow$ player team $= h$,
  is-home $=$ true, opponent difficulty $= d_h$
- $(\text{Gameweek}=w,\ \text{OppKey}=h) \Rightarrow$ player team $= a$,
  is-home $=$ false, opponent difficulty $= d_a$

Each player–gameweek row is then joined on
`(Gameweek, opponent_team) → (Gameweek, OppKey)`, which deterministically
yields the player's actual team for that fixture. This approach is
robust to mid-season transfers and double-gameweeks alike.

### 2.4 Player Identity Resolution

Names are normalised by (i) lower-casing, (ii) Unicode-decomposing
accented characters via `unidecode`, (iii) stripping numeric suffixes
that occasionally appear in raw files (e.g. `"aaron cresswell 376"`),
(iv) replacing underscores with spaces, (v) retaining only alphanumeric
and whitespace characters, and (vi) collapsing repeated whitespace.
A persistent UUID-v4 is assigned per normalised-name on first
observation and re-used on subsequent runs by reading back the saved
mapping file.

A symmetric failure mode to transliteration collapse is **homonym
collision** — distinct players who happen to share a normalised name.
The Premier League's history contains multiple instances of this, the
most notable being two footballers named *Ben Davies* who were
simultaneously on the Liverpool FC roster during the 2020-21 season.
To guard against such cases, a final identity pass verifies the
invariant that every UUID must correspond to exactly one FPL
``element`` (``Code``) value. UUIDs that violate this invariant are
split: the ``Code`` with the greatest number of associated rows
retains the original UUID (preserving the stability of the majority
player's history across runs), while minority ``Code`` values receive
newly-minted UUIDs with their ``Player Name Norm`` field suffixed for
disambiguation. The three identity passes
(§4.10 — name-based assignment, §4.10b — Code-based reconciliation,
§4.10c — homonym splitting) are applied strictly *before* the
computation of rolling-form features so that no cross-player leakage
can occur.

### 2.5 Feature Engineering

Two leakage-safe features are computed for every player–gameweek row:

**Consecutive-absence flag.** A binary indicator that fires when the
player has recorded zero minutes for three or more consecutive
gameweeks. This is computed via a vectorised grouped operation rather
than a Python loop.

**Lagged rolling form.** For each metric in $\mathcal{M}$ and window
size $w \in \{3, 5\}$, the feature is defined as

$$\mathrm{Avg}\_{m,w}(p, t) =
  \frac{1}{w} \sum_{i=1}^{w} m\bigl(p, t - i\bigr)$$

where $m$ is the metric and $(p, t)$ is the (player, gameweek) index.
The summation explicitly excludes the current gameweek $t$ — implemented
in `pandas` as `groupby(...).shift(1).rolling(w).mean()` — to guarantee
that no information from the prediction target leaks into the predictor.
The metric set $\mathcal{M}$ is

$$\mathcal{M} = \{\text{Total Points},\ \text{Minutes Played},\
  \text{Goals Scored},\ \text{Assists},\ \text{Goals Conceded},\
  \text{ICT Index},\ \text{Threat},\ \text{Creativity},\
  \text{Influence}\}.$$

### 2.6 Output Specification

The artefact is `data/processed/historical_baseline.csv`, encoded as
UTF-8 with BOM for Excel compatibility. Each row is uniquely keyed by
the tuple $(\text{Player UUID},\ \text{season},\ \text{Gameweek})$.
Schema and dtypes are reported in Section 5.

**Handling of Double Gameweeks and source-CSV duplicates.** The
public FPL data archive contains two superficially similar but
semantically distinct row-duplication patterns. The first is
*Double Gameweeks* (DGWs) — a real-world scheduling artefact in which
a team plays two matches within a single FPL gameweek — where each of
a player's two appearances is a distinct, valid observation. The
second is *source-CSV duplication*, in which certain seasons' raw
files contain rows replicated verbatim due to aggregation issues in
the upstream archive. These must be handled asymmetrically: DGW rows
are preserved as two distinct observations, whereas source duplicates
are collapsed. The procedure deduplicates on the composite key
$(\text{Code},\ \text{season},\ \text{Gameweek},\ \text{Opponent ID})$,
which is unique for every real match; where multiple rows collide on
this key, the row with the greatest ``Minutes Played`` is retained.
A derived $\text{Fixture Index} \in \{1, 2, \ldots\}$ is then assigned
per $(\text{Code},\ \text{season},\ \text{Gameweek})$ group to make
DGW observations explicit for downstream consumers (particularly the
captaincy and auto-substitution logic of
``forecasting_and_backtest.ipynb``).

## 3. Environment and Dependencies

This section configures the runtime: imports are grouped by origin
(standard library, third-party, project-local), structured logging
replaces ad-hoc `print` debugging, deterministic seeding ensures the
UUID assignment fallback is reproducible, and pandas display options
are tuned for legible inline output.

In [3]:
"""Section 3.1 — Imports."""

# Standard library
from __future__ import annotations

import logging
import re
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

# Third-party
import numpy as np
import pandas as pd
from unidecode import unidecode

In [4]:
"""Section 3.2 — Logging configuration.

A single root logger is configured at INFO level with a concise format.
All status messages emitted by this notebook flow through this logger,
which replaces the ad-hoc ``print`` statements used during prototyping.
"""

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("historical_baseline")
log.info("Logger initialised.")

13:46:35 | INFO    | Logger initialised.


In [5]:
"""Section 3.3 — Reproducibility.

A fixed seed is set so that any UUID generation that occurs *without*
a pre-existing mapping file produces a deterministic ordering (the
``uuid4`` values themselves remain non-deterministic by design, but
the iteration order over normalised names is stable).
"""

RANDOM_SEED = 20260422
np.random.seed(RANDOM_SEED)
log.info("NumPy random seed fixed at %d.", RANDOM_SEED)

13:47:34 | INFO    | NumPy random seed fixed at 20260422.


In [6]:
"""Section 3.4 — Pandas display options."""

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

In [7]:
"""Section 3.5 — Path constants and run configuration.

All paths are resolved relative to the project root, which is located
by walking up from the current working directory until a sentinel file
(``master_team_list.csv``) is found. This avoids the silent-failure
mode of name-based root detection.
"""


def _find_project_root(start: Path, sentinel: str = "master_team_list.csv") -> Path:
    """Walk upward from ``start`` until ``sentinel`` is found.

    Parameters
    ----------
    start : Path
        Directory from which to begin the upward walk.
    sentinel : str
        Filename whose presence marks the project root.

    Returns
    -------
    Path
        The resolved project root directory.

    Raises
    ------
    FileNotFoundError
        If the sentinel is not located before reaching the filesystem root.
    """
    current = start.resolve()
    while True:
        if (current / sentinel).exists():
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Project root sentinel '{sentinel}' not found above {start}."
            )
        current = current.parent


PROJECT_ROOT = _find_project_root(Path.cwd())
DATA_ROOT = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output"
MASTER_TEAM_LIST_PATH = PROJECT_ROOT / "master_team_list.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Run configuration
HISTORICAL_SEASONS: tuple[str, ...] = (
    "2020-21",
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
)

# Output artefacts
HISTORICAL_BASELINE_PATH = PROCESSED_DIR / "historical_baseline.csv"
UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping.csv"
CLEANED_UUID_MAPPING_PATH = OUTPUT_DIR / "player_uuid_mapping_cleaned.csv"

# Domain constants
POSITION_MAP: dict[int, str] = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD"}

ROLLING_METRICS: tuple[str, ...] = (
    "Total Points",
    "Minutes Played",
    "Goals Scored",
    "Assists",
    "Goals Conceded",
    "ICT Index",
    "Threat",
    "Creativity",
    "Influence",
)
ROLLING_WINDOWS: tuple[int, ...] = (3, 5)
ABSENCE_STREAK_THRESHOLD = 3

log.info("Project root resolved to: %s", PROJECT_ROOT)
log.info("Historical seasons in scope: %s", ", ".join(HISTORICAL_SEASONS))
log.info("Output artefact: %s", HISTORICAL_BASELINE_PATH)

13:47:48 | INFO    | Project root resolved to: C:\Python\fpl_pipeline
13:47:48 | INFO    | Historical seasons in scope: 2020-21, 2021-22, 2022-23, 2023-24, 2024-25
13:47:48 | INFO    | Output artefact: C:\Python\fpl_pipeline\data\processed\historical_baseline.csv


## 4. Implementation

The pipeline is decomposed into a sequence of pure, single-responsibility
functions, each documented with a NumPy-style docstring. The
orchestration cell at the end of this section composes them into a
linear dataflow that can be read end-to-end in fewer than thirty lines.

A small `ValidationReport` dataclass is also defined to give the
downstream validation routine a structured return value rather than a
dictionary of loose strings.

In [8]:
"""Section 4.1 — Player-name normalisation utility."""


def normalize_player_name(name: object) -> object:
    """Return a canonical, comparable representation of a player name.

    The transformation chain is, in order:

    1. Coerce to ``str`` and strip outer whitespace, then lower-case.
    2. Unicode-decompose accented characters (e.g. ``"ø"`` → ``"o"``).
    3. Remove any trailing numeric suffix
       (e.g. ``"aaron cresswell 376"`` → ``"aaron cresswell"``).
    4. Replace underscores with spaces.
    5. Retain only alphanumeric and whitespace characters.
    6. Collapse repeated whitespace into single spaces.

    Parameters
    ----------
    name : object
        Raw player name. ``NaN`` is propagated unchanged.

    Returns
    -------
    object
        Normalised name string, or the original ``NaN`` if input was missing.
    """
    if pd.isna(name):
        return name

    text = str(name).strip().lower()
    text = unidecode(text)
    text = re.sub(r"[\s_]*\d+\s*$", "", text)
    text = text.replace("_", " ")
    text = "".join(ch for ch in text if ch.isalnum() or ch.isspace())
    text = " ".join(text.split())
    return text

In [9]:
"""Section 4.2 — Gameweek data ingestion."""

GAMEWEEK_KEEP_COLUMNS: tuple[str, ...] = (
    "name",
    "element",
    "minutes",
    "goals_scored",
    "assists",
    "clean_sheets",
    "goals_conceded",
    "yellow_cards",
    "red_cards",
    "total_points",
    "influence",
    "creativity",
    "threat",
    "ict_index",
    "opponent_team",
    "was_home",
)


def load_gameweek_data(
    seasons: Iterable[str],
    data_root: Path = DATA_ROOT,
    keep_columns: Iterable[str] = GAMEWEEK_KEEP_COLUMNS,
) -> pd.DataFrame:
    """Load and concatenate per-gameweek CSVs across the requested seasons.

    Files are read from ``{data_root}/{season}/gws/gw{N}.csv``. Seasons
    or files that are missing on disk are skipped with a warning rather
    than raising, allowing partial-data runs during development.

    Parameters
    ----------
    seasons : Iterable[str]
        Season identifiers to ingest, e.g. ``("2020-21", "2021-22")``.
    data_root : Path
        Root directory containing per-season subfolders.
    keep_columns : Iterable[str]
        Subset of raw columns to retain, intersected with what is
        actually present in each file.

    Returns
    -------
    pandas.DataFrame
        Concatenated gameweek frame with two added columns: ``season``
        (str) and ``Gameweek`` (Int64).

    Raises
    ------
    ValueError
        If no rows could be loaded from any season.
    """
    keep = list(keep_columns)
    season_frames: list[pd.DataFrame] = []

    for season in seasons:
        season_dir = data_root / season
        gws_dir = season_dir / "gws"

        if not gws_dir.is_dir():
            log.warning("No 'gws' directory for season %s — skipping.", season)
            continue

        gw_files = sorted(
            (p for p in gws_dir.iterdir() if p.name.startswith("gw") and p.suffix == ".csv"),
            key=lambda p: int(p.stem.replace("gw", "")),
        )
        log.info("Season %s: %d gameweek files found.", season, len(gw_files))

        per_gw: list[pd.DataFrame] = []
        for gw_path in gw_files:
            gw_number = int(gw_path.stem.replace("gw", ""))
            raw = pd.read_csv(gw_path)
            present = [c for c in keep if c in raw.columns]
            frame = raw.loc[:, present].copy()
            frame["season"] = season
            frame["Gameweek"] = gw_number
            per_gw.append(frame)

        if per_gw:
            season_frames.append(pd.concat(per_gw, ignore_index=True))

    if not season_frames:
        raise ValueError(
            "No gameweek data could be loaded — verify DATA_ROOT and season folders."
        )

    combined = pd.concat(season_frames, ignore_index=True)
    log.info(
        "Loaded %s rows across %d seasons.",
        f"{len(combined):,}",
        combined["season"].nunique(),
    )
    return combined

In [10]:
"""Section 4.3 — Player metadata ingestion."""


def load_player_metadata(
    seasons: Iterable[str],
    data_root: Path = DATA_ROOT,
) -> dict[str, pd.DataFrame]:
    """Load ``players_raw.csv`` for each season and return a per-season dict.

    The ``id`` column is renamed to ``element`` for cross-season
    consistency. Only the metadata columns required downstream
    (``element_type``, ``web_name``, ``first_name``, ``second_name``)
    are retained; the unreliable ``team`` column is intentionally
    excluded — true team identification is performed in
    :func:`derive_team_from_fixtures`.

    Parameters
    ----------
    seasons : Iterable[str]
        Season identifiers to ingest.
    data_root : Path
        Root directory containing per-season subfolders.

    Returns
    -------
    dict[str, pandas.DataFrame]
        Mapping ``season -> dataframe`` indexed on ``element``.
    """
    out: dict[str, pd.DataFrame] = {}
    keep = ["element", "element_type", "web_name", "first_name", "second_name"]

    for season in seasons:
        path = data_root / season / "players_raw.csv"
        if not path.exists():
            log.warning("No players_raw.csv for %s — skipping.", season)
            continue

        raw = pd.read_csv(path)
        if "id" in raw.columns and "element" not in raw.columns:
            raw = raw.rename(columns={"id": "element"})

        present = [c for c in keep if c in raw.columns]
        frame = raw.loc[:, present].copy()
        frame["element"] = pd.to_numeric(frame["element"], errors="coerce").astype("Int64")

        out[season] = frame.set_index("element")
        log.info("Season %s: %d players in metadata table.", season, len(frame))

    return out

In [11]:
"""Section 4.4 — Team metadata ingestion."""


def _load_master_team_table(path: Path) -> pd.DataFrame | None:
    """Load the cross-season fallback team table, if present."""
    if not path.exists():
        return None
    table = pd.read_csv(path)
    table.columns = [c.lower() for c in table.columns]
    return table


_MASTER_TEAMS_DF = _load_master_team_table(MASTER_TEAM_LIST_PATH)


def load_team_metadata(season: str, data_root: Path = DATA_ROOT) -> pd.DataFrame:
    """Return team metadata for a single season.

    Reads from ``{data_root}/{season}/teams.csv`` when available and
    falls back to the project-level ``master_team_list.csv`` when not
    (this is the case for older seasons in the public mirror).

    Parameters
    ----------
    season : str
        Season identifier.
    data_root : Path
        Root directory containing per-season subfolders.

    Returns
    -------
    pandas.DataFrame
        DataFrame with columns ``["Team ID", "Team Name", "short_name"]``.

    Raises
    ------
    RuntimeError
        If neither the per-season nor the master table contains the
        requested season.
    """
    season_path = data_root / season / "teams.csv"

    if season_path.exists():
        raw = pd.read_csv(season_path)
        id_col = "id" if "id" in raw.columns else "code"
        renamed = raw.rename(columns={id_col: "Team ID", "name": "Team Name"})
        renamed["Team ID"] = pd.to_numeric(renamed["Team ID"], errors="coerce").astype("Int64")
        if "short_name" not in renamed.columns:
            renamed["short_name"] = renamed["Team Name"]
        return renamed[["Team ID", "Team Name", "short_name"]]

    if _MASTER_TEAMS_DF is not None:
        sub = _MASTER_TEAMS_DF[_MASTER_TEAMS_DF["season"] == season].copy()
        if not sub.empty:
            sub = sub.rename(columns={"team": "Team ID", "team_name": "Team Name"})
            sub["Team ID"] = pd.to_numeric(sub["Team ID"], errors="coerce").astype("Int64")
            sub["short_name"] = sub["Team Name"]
            log.info("Season %s: using master_team_list fallback.", season)
            return sub[["Team ID", "Team Name", "short_name"]]

    raise RuntimeError(f"No team metadata available for season {season}.")

In [12]:
"""Section 4.5 — Fixture-derived team identification (methodological core).

For each fixture in a given season, two long-format rows are emitted:
one keyed by the away team as ``OppKey`` (which yields the home team as
the player team), and the symmetric mirror. Joining player–gameweek
rows against this table on ``(Gameweek, opponent_team) == (Gameweek,
OppKey)`` deterministically recovers each player's actual team for
each match, including for mid-season transfers.
"""


def _build_fixture_long(season: str, data_root: Path = DATA_ROOT) -> pd.DataFrame:
    """Construct the long-format fixture table for one season."""
    fx_path = data_root / season / "fixtures.csv"
    if not fx_path.exists():
        raise FileNotFoundError(f"fixtures.csv missing for season {season}.")

    fx_raw = pd.read_csv(fx_path)
    gw_col = "event" if "event" in fx_raw.columns else "round"
    fx_raw[gw_col] = pd.to_numeric(fx_raw[gw_col], errors="coerce")
    fx_raw = fx_raw.dropna(subset=[gw_col, "team_h", "team_a"])

    rows: list[dict] = []
    for _, r in fx_raw.iterrows():
        gw = int(r[gw_col])
        h, a = int(r["team_h"]), int(r["team_a"])
        dh = int(r["team_h_difficulty"]) if pd.notna(r["team_h_difficulty"]) else 0
        da = int(r["team_a_difficulty"]) if pd.notna(r["team_a_difficulty"]) else 0
        rows.append({
            "Gameweek": gw, "OppKey": a,
            "Player Team ID": h, "Opponent ID": a,
            "Is Home": True, "Opponent Difficulty": dh,
        })
        rows.append({
            "Gameweek": gw, "OppKey": h,
            "Player Team ID": a, "Opponent ID": h,
            "Is Home": False, "Opponent Difficulty": da,
        })

    long = pd.DataFrame(rows)
    long["Gameweek"] = long["Gameweek"].astype("Int64")
    long["OppKey"] = long["OppKey"].astype("Int64")
    return long


def derive_team_from_fixtures(
    season_gw_frame: pd.DataFrame,
    season: str,
    metadata_by_season: dict[str, pd.DataFrame],
    data_root: Path = DATA_ROOT,
) -> pd.DataFrame:
    """Attach team and opponent identities/names to a single-season frame.

    Parameters
    ----------
    season_gw_frame : pandas.DataFrame
        Gameweek rows for a single season, as produced by
        :func:`load_gameweek_data` filtered to that season.
    season : str
        Season identifier.
    metadata_by_season : dict[str, pandas.DataFrame]
        Output of :func:`load_player_metadata`.
    data_root : Path
        Root directory containing per-season subfolders.

    Returns
    -------
    pandas.DataFrame
        Input frame enriched with ``Player Team ID``, ``Opponent ID``,
        ``Is Home``, ``Opponent Difficulty``, ``Player Team Name``,
        ``Opponent Name``, plus the metadata fields ``element_type``,
        ``web_name``, ``first_name`` and ``second_name``.
    """
    log.info("Building core frame for season %s …", season)

    df = season_gw_frame.copy()
    df["element"] = pd.to_numeric(df["element"], errors="coerce").astype("Int64")
    df["Gameweek"] = pd.to_numeric(df["Gameweek"], errors="coerce").astype("Int64")
    df["opponent_team"] = pd.to_numeric(df["opponent_team"], errors="coerce").astype("Int64")

    # Player metadata join (NB: team intentionally excluded — see Section 2.3).
    meta = metadata_by_season.get(season)
    if meta is not None:
        meta_cols = [c for c in ("element_type", "web_name", "first_name", "second_name") if c in meta.columns]
        df = df.merge(meta[meta_cols], left_on="element", right_index=True, how="left")

    # Fixture-derived team identification.
    fixture_long = _build_fixture_long(season, data_root=data_root)
    df = df.merge(
        fixture_long,
        left_on=["Gameweek", "opponent_team"],
        right_on=["Gameweek", "OppKey"],
        how="left",
    ).drop(columns=["OppKey"])

    # Human-readable team names.
    teams = load_team_metadata(season, data_root=data_root)
    df = df.merge(
        teams[["Team ID", "Team Name"]],
        left_on="Player Team ID",
        right_on="Team ID",
        how="left",
    ).rename(columns={"Team Name": "Player Team Name"}).drop(columns=["Team ID"])

    df = df.merge(
        teams.rename(columns={"Team ID": "Opponent ID", "Team Name": "Opponent Name"})[
            ["Opponent ID", "Opponent Name"]
        ],
        on="Opponent ID",
        how="left",
    )

    # Composite Player Name (first + second, falling back to raw `name`).
    if {"first_name", "second_name"}.issubset(df.columns):
        composed = (
            df["first_name"].fillna("") + " " + df["second_name"].fillna("")
        ).str.strip().replace("", np.nan)
        df["Player Name"] = composed.fillna(df["name"])
    else:
        df["Player Name"] = df["name"]

    df["Web Name"] = df.get("web_name")
    return df

In [13]:
"""Section 4.6 — Canonical column renaming and position assignment."""

CANONICAL_RENAME: dict[str, str] = {
    "element": "Code",
    "minutes": "Minutes Played",
    "goals_scored": "Goals Scored",
    "assists": "Assists",
    "clean_sheets": "Clean Sheet",
    "goals_conceded": "Goals Conceded",
    "yellow_cards": "Yellow Card",
    "red_cards": "Red Cards",
    "total_points": "Total Points",
    "influence": "Influence",
    "creativity": "Creativity",
    "threat": "Threat",
    "ict_index": "ICT Index",
}


def assign_canonical_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Apply canonical column names and derive ``Position`` from ``element_type``.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame produced by :func:`derive_team_from_fixtures`, concatenated
        across seasons.

    Returns
    -------
    pandas.DataFrame
        Frame with renamed columns, an added ``Position`` column
        (``GK``, ``DEF``, ``MID`` or ``FWD``), and a normalised
        ``Player Name Norm`` column.
    """
    out = df.rename(columns=CANONICAL_RENAME).copy()

    if "element_type" in out.columns:
        out["Position"] = out["element_type"].map(POSITION_MAP)
    else:
        out["Position"] = np.nan

    out["Player Name Norm"] = out["Player Name"].apply(normalize_player_name)
    return out

In [14]:
"""Section 4.7 — Cross-season position fix-up.

Some rows arrive without a position because the per-season
``players_raw.csv`` did not contain the player at the time of the
snapshot (typically late-season transfers). This routine searches
across all loaded seasons' metadata for any record of the player's
``element_type`` and back-fills missing positions.
"""


def repair_missing_positions(
    df: pd.DataFrame,
    metadata_by_season: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    """Back-fill ``Position`` from any season's metadata where available.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame with a ``Code`` column (player ``element`` id) and a
        possibly-missing ``Position`` column.
    metadata_by_season : dict[str, pandas.DataFrame]
        Output of :func:`load_player_metadata`.

    Returns
    -------
    pandas.DataFrame
        Frame with ``Position`` repaired where possible. Any rows that
        remain unresolved are logged but not removed at this stage.
    """
    if not metadata_by_season:
        return df

    pieces: list[pd.DataFrame] = []
    for season, meta in metadata_by_season.items():
        if "element_type" not in meta.columns:
            continue
        slice_ = meta.reset_index()[["element", "element_type"]].copy()
        pieces.append(slice_)

    if not pieces:
        return df

    combined = (
        pd.concat(pieces, ignore_index=True)
        .dropna(subset=["element"])
        .drop_duplicates(subset="element", keep="first")
    )
    combined["element"] = pd.to_numeric(combined["element"], errors="coerce").astype("Int64")
    code_to_type = dict(zip(combined["element"], combined["element_type"]))

    repaired = df.copy()
    if "Code" not in repaired.columns:
        return repaired

    needs_fix = repaired["Position"].isna()
    if not needs_fix.any():
        return repaired

    fallback_type = repaired.loc[needs_fix, "Code"].map(code_to_type)
    repaired.loc[needs_fix, "Position"] = fallback_type.map(POSITION_MAP)

    remaining = repaired["Position"].isna().sum()
    log.info(
        "Position repair: filled %d of %d missing rows; %d remain unresolved.",
        int(needs_fix.sum() - remaining),
        int(needs_fix.sum()),
        int(remaining),
    )
    return repaired

In [53]:
"""Section 4.7b — Source-CSV deduplication and DGW fixture indexing.

The public FPL data archive contains two distinct categories of rows
that share the same ``(Player, season, Gameweek)`` tuple:

1. **Legitimate Double-Gameweek (DGW) rows.** A player plays two
   distinct matches in one gameweek; the two rows have *different*
   ``Opponent ID`` values. These rows are real observations and must
   be preserved.

2. **Source-CSV duplicates.** Some seasons' raw files contain rows
   that are replicated verbatim (same opponent, same stats). These
   are archive artefacts that artificially inflate per-player totals
   and rolling averages.

This routine separates the two cases by deduplicating on the composite
key ``(Code, season, Gameweek, Opponent ID)`` — which is unique for
every real match — and assigns a per-group ``Fixture Index`` so that
DGW rows become explicit to downstream consumers. Where two rows share
the full composite key (i.e. source duplicates), the row with the
greatest ``Minutes Played`` is retained (ties broken by ``Total
Points``), so a "played" row always wins over a "did-not-play" row.
"""


def deduplicate_and_index_fixtures(df: pd.DataFrame) -> pd.DataFrame:
    """Remove source-CSV duplicates and assign a per-match fixture index.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame produced after canonical schema assignment. Must contain
        ``Code``, ``season``, ``Gameweek``, ``Opponent ID``,
        ``Minutes Played`` and ``Total Points`` columns.

    Returns
    -------
    pandas.DataFrame
        Deduplicated frame with an added ``Fixture Index`` column
        (``Int64``, 1-based) uniquely numbering the matches of each
        ``(Code, season, Gameweek)`` group.
    """
    n_before = len(df)

    # Rank within each composite-key group, preferring rows with the
    # greatest Minutes Played (ties broken by Total Points, then by
    # original index for full determinism).
    work = df.copy()
    work["_minutes_num"] = pd.to_numeric(work["Minutes Played"], errors="coerce").fillna(-1)
    work["_points_num"] = pd.to_numeric(work["Total Points"], errors="coerce").fillna(-1)
    work["_tiebreak"] = np.arange(len(work))

    key = ["Code", "season", "Gameweek", "Opponent ID"]
    work = work.sort_values(
        by=key + ["_minutes_num", "_points_num", "_tiebreak"],
        ascending=[True, True, True, True, False, False, True],
        kind="mergesort",
    )
    deduped = work.drop_duplicates(subset=key, keep="first").drop(
        columns=["_minutes_num", "_points_num", "_tiebreak"]
    )

    n_dropped = n_before - len(deduped)
    if n_dropped > 0:
        dropped_by_season = (
            df.loc[~df.index.isin(deduped.index)]
            .groupby("season")
            .size()
            .to_dict()
        )
        log.info(
            "Dropped %s source-CSV duplicate rows (by season: %s).",
            f"{n_dropped:,}",
            dropped_by_season,
        )
    else:
        log.info("No source-CSV duplicates detected.")

    # Assign Fixture Index within each (Code, season, Gameweek) group.
    deduped = deduped.sort_values(
        ["Code", "season", "Gameweek", "Opponent ID"], kind="mergesort"
    )
    deduped["Fixture Index"] = (
        deduped.groupby(["Code", "season", "Gameweek"]).cumcount() + 1
    ).astype("Int64")

    # Report DGW prevalence for transparency.
    dgw_rows = deduped[deduped["Fixture Index"] > 1]
    if len(dgw_rows) > 0:
        dgw_summary = dgw_rows.groupby("season").size().to_dict()
        log.info(
            "Retained %s Double-Gameweek rows (by season: %s).",
            f"{len(dgw_rows):,}",
            dgw_summary,
        )

    return deduped.reset_index(drop=True)

In [54]:
"""Section 4.8 — Vectorised consecutive-absence flag.

Replaces the per-player Python loop used in the prototype with a
grouped cumulative-count construction. Within each player's history,
a new "absence run" is started whenever a non-zero-minute row is
observed; a cumulative count within that run gives the streak length.
The flag fires when the streak reaches the configured threshold.
"""


def flag_consecutive_absences(
    df: pd.DataFrame,
    threshold: int = ABSENCE_STREAK_THRESHOLD,
) -> pd.DataFrame:
    """Add an ``Injury/Unavailable`` indicator column.

    The flag equals 1 on row :math:`(p, t)` iff the player has accumulated
    at least ``threshold`` consecutive zero-minute appearances ending at
    gameweek :math:`t` (inclusive). The streak is computed across the
    chronological order ``(season, Gameweek)`` per normalised player name.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame containing ``Player Name Norm``, ``season``, ``Gameweek``
        and ``Minutes Played`` columns.
    threshold : int
        Minimum consecutive zero-minute count required to fire the flag.

    Returns
    -------
    pandas.DataFrame
        Sorted copy of ``df`` with an added ``Injury/Unavailable``
        ``int8`` column.
    """
    out = df.sort_values(["Player Name Norm", "season", "Gameweek"]).copy()
    minutes = pd.to_numeric(out["Minutes Played"], errors="coerce").fillna(0)

    is_zero = (minutes == 0).astype(int)

    # Run-id increments each time a non-zero row is observed, partitioning
    # each player's series into maximal zero-minute runs.
    run_break = (minutes != 0).astype(int)
    run_id = run_break.groupby(out["Player Name Norm"]).cumsum()

    # Within each (player, run_id), count consecutive zero rows.
    streak = (
        is_zero.groupby([out["Player Name Norm"], run_id]).cumsum()
    )

    out["Injury/Unavailable"] = (streak >= threshold).astype("int8")
    return out

In [55]:
"""Section 4.9 — Lagged rolling form features.

For each metric in ``ROLLING_METRICS`` and each window in
``ROLLING_WINDOWS``, computes the per-player rolling mean *strictly
prior* to the current row. The ``shift(1)`` ensures no leakage from
the prediction target. ``min_periods=1`` allows partial windows at
the start of a player's history rather than emitting ``NaN``.
"""


def compute_rolling_features(
    df: pd.DataFrame,
    metrics: Iterable[str] = ROLLING_METRICS,
    windows: Iterable[int] = ROLLING_WINDOWS,
) -> pd.DataFrame:
    """Append leakage-safe rolling-mean features per (player, season).

    Parameters
    ----------
    df : pandas.DataFrame
        Sorted player–gameweek frame.
    metrics : Iterable[str]
        Column names whose rolling means should be computed.
    windows : Iterable[int]
        Window sizes in gameweeks.

    Returns
    -------
    pandas.DataFrame
        Input frame with one new ``Avg_<metric>_L<window>`` column per
        (metric, window) pair. Resets within each season to avoid
        bleeding form across summer breaks.
    """
    out = df.sort_values(["Player Name Norm", "season", "Gameweek"]).copy()
    grouper = out.groupby(["Player Name Norm", "season"], sort=False)

    for window in windows:
        for metric in metrics:
            if metric not in out.columns:
                continue
            new_col = f"Avg_{metric}_L{window}"
            out[new_col] = (
                grouper[metric]
                .transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
                .fillna(0.0)
            )
    return out

In [56]:
"""Section 4.10 — Stable cross-season player UUID assignment.

The curated mapping file (``CLEANED_UUID_MAPPING_PATH``), when present,
takes precedence for any name it contains — this is where ambiguous
duplicate-name resolutions are recorded by hand. Names *not* covered by
the curated mapping fall back to the auto-managed mapping
(``UUID_MAPPING_PATH``); newly-observed names are appended to the auto
mapping with freshly-generated UUIDs and persisted, so that no row in
the artefact is ever left with a null ``Player UUID``.
"""


def assign_player_uuids(
    df: pd.DataFrame,
    cleaned_path: Path = CLEANED_UUID_MAPPING_PATH,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Attach a stable ``Player UUID`` to every row.

    The function maintains a two-tier mapping. Tier 1 is the curated
    file at ``cleaned_path`` (manual disambiguations, never modified).
    Tier 2 is the auto file at ``auto_path``, which is read, extended
    with UUIDs for any newly-observed normalised names, and rewritten.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame containing ``Player Name Norm``, ``Player Name`` and
        ``Web Name`` columns.
    cleaned_path : Path
        Curated mapping; entries here override the auto mapping.
    auto_path : Path
        Auto-managed mapping; created or extended as needed.

    Returns
    -------
    pandas.DataFrame
        Input frame with an added ``Player UUID`` column (string). All
        non-null ``Player Name Norm`` values are guaranteed to receive
        a UUID.
    """
    out = df.copy()

    # --- Tier 1: curated mapping (highest priority) -------------------
    curated_lookup: dict[str, str] = {}
    if cleaned_path.exists():
        curated = pd.read_csv(cleaned_path)
        norm_col = next(c for c in curated.columns if c.lower() == "player name norm")
        uuid_col = next(c for c in curated.columns if c.lower() == "player uuid")
        curated_lookup = dict(
            zip(curated[norm_col].astype(str), curated[uuid_col].astype(str))
        )
        log.info("Loaded %d curated UUID entries from %s.", len(curated_lookup), cleaned_path)

    # --- Tier 2: auto mapping (extended with new names) ---------------
    auto_lookup: dict[str, str] = {}
    if auto_path.exists():
        existing = pd.read_csv(auto_path)
        if {"Player Name Norm", "Player UUID"}.issubset(existing.columns):
            auto_lookup = dict(
                zip(existing["Player Name Norm"].astype(str), existing["Player UUID"].astype(str))
            )
        log.info("Loaded %d existing UUID entries from %s.", len(auto_lookup), auto_path)

    # Identify names present in the data that need an auto-UUID.
    observed_names = sorted(out["Player Name Norm"].dropna().astype(str).unique())
    newly_minted = 0
    for name in observed_names:
        if name in curated_lookup:
            continue                         # tier 1 wins
        if name not in auto_lookup:
            auto_lookup[name] = str(uuid.uuid4())
            newly_minted += 1

    if newly_minted:
        log.info("Minted %d new UUIDs for previously-unseen normalised names.", newly_minted)

    # Persist the (possibly extended) auto mapping. Curated file is never written.
    representative = (
        out.dropna(subset=["Player Name Norm"])
        .groupby("Player Name Norm")[["Player Name", "Web Name"]]
        .agg(lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan)
        .reset_index()
    )
    persisted = (
        pd.DataFrame(
            {"Player Name Norm": list(auto_lookup.keys()),
             "Player UUID": list(auto_lookup.values())}
        )
        .merge(representative, on="Player Name Norm", how="left")
    )
    auto_path.parent.mkdir(parents=True, exist_ok=True)
    persisted.to_csv(auto_path, index=False, encoding="utf-8-sig")
    log.info("Persisted %d total UUID entries to %s.", len(persisted), auto_path)

    # Apply: tier 1 overrides tier 2.
    final_lookup = {**auto_lookup, **curated_lookup}
    out["Player UUID"] = out["Player Name Norm"].map(final_lookup)

    null_count = out["Player UUID"].isna().sum()
    if null_count:
        log.warning(
            "%d rows have null Player UUID (likely missing Player Name Norm).",
            null_count,
        )
    else:
        log.info("All rows successfully assigned a Player UUID.")

    return out

In [73]:
"""Section 4.10b — Code-based UUID reconciliation.

When two name variants of the same player produce two distinct
``Player Name Norm`` values (and therefore two distinct UUIDs), they
can still be reconciled because the FPL platform assigns a single
``element`` integer per player per season. Two name variants observed
under the same ``(season, element)`` are by construction the same
person; their UUIDs are merged, with the alphabetically-lower UUID
chosen as the canonical survivor (a deterministic tie-break that is
independent of run order).

This routine also rewrites both the in-memory frame and the
auto-managed mapping file so that subsequent runs inherit the
reconciliation.
"""


def reconcile_uuids_via_code(
    df: pd.DataFrame,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Merge UUIDs that map to the same FPL ``element`` within a season.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame with ``Player UUID``, ``Code`` (FPL ``element`` id) and
        ``season`` columns.
    auto_path : Path
        Auto-managed UUID mapping file; will be rewritten with the
        post-reconciliation mapping.

    Returns
    -------
    pandas.DataFrame
        Frame with ``Player UUID`` rewritten so that all rows of the
        same underlying player share a single UUID.
    """
    out = df.copy()

    # Build the equivalence relation: pairs of UUIDs that co-occur
    # under the same (season, Code).
    pairs = (
        out.dropna(subset=["Player UUID", "Code"])
        .groupby(["season", "Code"])["Player UUID"]
        .unique()
    )

    # Union-Find over UUIDs.
    parent: dict[str, str] = {}

    def find(x: str) -> str:
        while parent.get(x, x) != x:
            parent[x] = parent.get(parent[x], parent[x])
            x = parent[x]
        return x

    def union(a: str, b: str) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        # Deterministic tie-break: lexicographically smaller UUID wins.
        winner, loser = (ra, rb) if ra < rb else (rb, ra)
        parent[loser] = winner

    for uuid_set in pairs:
        if len(uuid_set) < 2:
            continue
        canonical = min(uuid_set)
        for other in uuid_set:
            if other != canonical:
                union(canonical, other)

    # Build the rewrite map: every UUID -> its canonical representative.
    all_uuids = out["Player UUID"].dropna().astype(str).unique()
    rewrite = {u: find(u) if u in parent else u for u in all_uuids}

    n_merged = sum(1 for u, v in rewrite.items() if u != v)
    if n_merged == 0:
        log.info("UUID reconciliation: no transliteration collisions detected.")
        return out

    log.info(
        "UUID reconciliation: collapsing %d aliased UUIDs into their canonical forms.",
        n_merged,
    )
    out["Player UUID"] = out["Player UUID"].map(rewrite).fillna(out["Player UUID"])

    # Update the persisted auto mapping so subsequent runs are stable.
    if auto_path.exists():
        persisted = pd.read_csv(auto_path)
        if "Player UUID" in persisted.columns:
            persisted["Player UUID"] = (
                persisted["Player UUID"].map(rewrite).fillna(persisted["Player UUID"])
            )
            # Drop now-duplicate rows (same Norm name pointing at same canonical UUID).
            persisted = persisted.drop_duplicates(subset=["Player Name Norm"], keep="first")
            persisted.to_csv(auto_path, index=False, encoding="utf-8-sig")
            log.info("Rewrote auto UUID mapping at %s.", auto_path)

    return out

In [74]:
"""Section 4.10c — Homonym resolution via Code-based UUID splitting.

The name-based UUID assignment of §4.10 and the transliteration-merge
of §4.10b collectively ensure that *the same player* carries a single
UUID across seasons — but neither guards against the opposite failure
mode: distinct players who happen to share a normalised name
(homonyms) being assigned a single UUID.

Concrete real-world example. In the 2020-21 January transfer window,
Liverpool signed two players named *Ben Davies* — one from Tottenham
Hotspur (FPL ``Code`` 248) and one from Preston North End (``Code``
364). Both produce the normalised form ``"ben davies"`` and would
therefore collapse onto a single UUID, causing their career rows to
be pooled downstream.

This routine detects such collisions — any UUID that corresponds to
more than one FPL ``Code`` — and splits them. The UUID associated
with the ``Code`` that has the greatest number of rows retains the
original identifier (preserving historical stability for the
majority player); minority ``Code`` values receive freshly-minted
UUIDs. The auto-managed mapping file is extended so that the split
is persisted across runs.
"""


def split_homonym_uuids(
    df: pd.DataFrame,
    auto_path: Path = UUID_MAPPING_PATH,
) -> pd.DataFrame:
    """Split UUIDs that collapse distinct FPL ``Code`` values.

    Parameters
    ----------
    df : pandas.DataFrame
        Frame with ``Player UUID``, ``Code``, ``Player Name``,
        ``Player Name Norm`` and ``Web Name`` columns.
    auto_path : Path
        Auto-managed UUID mapping file; augmented with fresh
        ``{normalised_name}#<n>`` entries where homonyms were split.

    Returns
    -------
    pandas.DataFrame
        Frame with ``Player UUID`` and ``Player Name Norm`` rewritten
        so that every distinct ``Code`` carries a distinct UUID.
    """
    out = df.copy()
    non_null = out.dropna(subset=["Player UUID", "Code"])

    uuid_to_codes = (
        non_null.groupby("Player UUID")["Code"].unique()
    )
    offenders = uuid_to_codes[uuid_to_codes.apply(len) > 1]
    if offenders.empty:
        log.info("Homonym split: no UUIDs span multiple FPL Codes.")
        return out

    log.warning(
        "Homonym split: %d UUID(s) span multiple FPL Codes and will be split.",
        len(offenders),
    )

    # Accumulate fresh entries for persistence.
    new_mapping_rows: list[dict] = []

    for original_uuid, codes in offenders.items():
        # Rank the Codes by row count within this UUID; the majority
        # Code retains the original UUID.
        subset = non_null[non_null["Player UUID"] == original_uuid]
        code_counts = subset["Code"].value_counts()
        majority_code = code_counts.index[0]
        minority_codes = code_counts.index[1:].tolist()

        # For each minority Code, mint a new UUID and a disambiguated
        # normalised-name suffix so the persisted mapping remains
        # unique-keyed.
        for rank, minor_code in enumerate(minority_codes, start=1):
            new_uuid = str(uuid.uuid4())
            mask = (out["Player UUID"] == original_uuid) & (out["Code"] == minor_code)
            # Derive a disambiguated norm by appending "#n" — this is
            # internal only; the human-readable Player Name is unchanged.
            representative = out.loc[mask].iloc[0]
            disambiguated_norm = f"{representative['Player Name Norm']}#{rank}"
            out.loc[mask, "Player UUID"] = new_uuid
            out.loc[mask, "Player Name Norm"] = disambiguated_norm

            new_mapping_rows.append({
                "Player Name Norm": disambiguated_norm,
                "Player UUID": new_uuid,
                "Player Name": representative["Player Name"],
                "Web Name": representative.get("Web Name"),
            })

            log.info(
                "  Split '%s' (Code %s): %d rows reassigned to new UUID %s.",
                representative["Player Name"],
                minor_code,
                int(mask.sum()),
                new_uuid,
            )

    # Persist the new disambiguated entries alongside the existing mapping.
    if new_mapping_rows and auto_path.exists():
        existing = pd.read_csv(auto_path)
        augmented = pd.concat(
            [existing, pd.DataFrame(new_mapping_rows)],
            ignore_index=True,
        ).drop_duplicates(subset=["Player Name Norm"], keep="first")
        augmented.to_csv(auto_path, index=False, encoding="utf-8-sig")
        log.info(
            "Persisted %d new homonym-split entries to %s.",
            len(new_mapping_rows),
            auto_path,
        )

    return out

In [88]:
"""Section 4.11 — Final column projection and persistence."""

BASE_COLUMNS: tuple[str, ...] = (
    "Player UUID",
    "Code",
    "Player Name",
    "Web Name",
    "Player Team ID",
    "Player Team Name",
    "season",
    "Gameweek",
    "Fixture Index",
    "Minutes Played",
    "Goals Scored",
    "Assists",
    "Clean Sheet",
    "Goals Conceded",
    "Yellow Card",
    "Red Cards",
    "Total Points",
    "Threat",
    "ICT Index",
    "Influence",
    "Creativity",
    "Opponent ID",
    "Opponent Name",
    "Opponent Difficulty",
    "Is Home",
    "Position",
    "Injury/Unavailable",
)


def project_and_persist(df: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    """Select the canonical column ordering and write the artefact."""
    base_present = [c for c in BASE_COLUMNS if c in df.columns]
    lagged_cols = sorted(c for c in df.columns if c.startswith("Avg_"))
    final_cols = base_present + lagged_cols

    projected = df.loc[:, final_cols].copy()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    projected.to_csv(output_path, index=False, encoding="utf-8-sig")

    log.info(
        "Wrote %s rows × %d columns to %s.",
        f"{len(projected):,}",
        projected.shape[1],
        output_path,
    )
    return projected

In [89]:
"""Section 4.12 — Structured validation report."""


@dataclass
class ValidationReport:
    """Lightweight container for dataset diagnostic results."""

    total_rows: int = 0
    seasons: list[str] = field(default_factory=list)
    duplicate_key_rows: int = 0
    dgw_rows: int = 0
    missing_opponent_difficulty: int = 0
    unresolved_positions: int = 0
    null_player_uuid: int = 0
    rows_per_season: pd.Series = field(default_factory=lambda: pd.Series(dtype=int))
    completeness: pd.DataFrame = field(default_factory=pd.DataFrame)

    def as_summary_table(self) -> pd.DataFrame:
        """Return a one-column dataframe summarising the headline metrics."""
        return pd.DataFrame(
            {
                "Value": [
                    self.total_rows,
                    ", ".join(self.seasons),
                    self.duplicate_key_rows,
                    self.dgw_rows,
                    self.missing_opponent_difficulty,
                    self.unresolved_positions,
                    self.null_player_uuid,
                ]
            },
            index=[
                "Total rows",
                "Seasons covered",
                "Duplicate (UUID, season, GW, Opponent) rows",
                "Legitimate DGW rows (Fixture Index > 1)",
                "Missing Opponent Difficulty",
                "Unresolved Position rows",
                "Null Player UUID rows",
            ],
        )


def validate_dataset(df: pd.DataFrame) -> ValidationReport:
    """Compute structured data-quality diagnostics for the artefact."""
    report = ValidationReport()
    report.total_rows = len(df)
    report.seasons = sorted(df["season"].unique().tolist())

    # The correct composite key: (UUID, season, GW, Opponent ID).
    # DGW rows are legitimate because they have different Opponent IDs.
    key = ["Player UUID", "season", "Gameweek", "Opponent ID"]
    report.duplicate_key_rows = int(df.duplicated(subset=key, keep=False).sum())

    if "Fixture Index" in df.columns:
        report.dgw_rows = int((df["Fixture Index"] > 1).sum())

    report.missing_opponent_difficulty = int(df["Opponent Difficulty"].isna().sum())
    report.unresolved_positions = int(df["Position"].isna().sum())
    report.null_player_uuid = int(df["Player UUID"].isna().sum())
    report.rows_per_season = df.groupby("season").size().rename("rows")

    completeness = (
        df.notna().groupby(df["season"]).mean().T.mul(100).round(2)
    )
    completeness.columns.name = "season"
    completeness.index.name = "column"
    report.completeness = completeness
    return report

### 4.13 Pipeline Orchestration

The cell below composes every function defined above into a single
linear pipeline. Each step is annotated with the section that defines
its semantics, so a reader of the notebook can navigate from
orchestration to specification in one click.

In [90]:
"""Section 4.13 — Orchestration."""

# 1. Ingest raw gameweek + metadata for the historical seasons (§4.2, §4.3).
gameweek_data = load_gameweek_data(HISTORICAL_SEASONS)
player_metadata = load_player_metadata(HISTORICAL_SEASONS)

# 2. Per-season fixture-derived team identification (§4.5), then concatenate.
season_cores = [
    derive_team_from_fixtures(
        gameweek_data[gameweek_data["season"] == season],
        season=season,
        metadata_by_season=player_metadata,
    )
    for season in HISTORICAL_SEASONS
]
core_frame = pd.concat(season_cores, ignore_index=True)

# 3. Canonical schema and position assignment (§4.6), then cross-season repair (§4.7).
core_frame = assign_canonical_schema(core_frame)
core_frame = repair_missing_positions(core_frame, player_metadata)

# 4. Source-CSV deduplication + DGW indexing (§4.7b) — MUST precede feature
#    engineering so that rolling averages and absence flags operate on
#    clean, DGW-aware rows.
core_frame = deduplicate_and_index_fixtures(core_frame)

# 5. Identity pipeline (§4.10 → §4.10b → §4.10c). Run BEFORE feature
#    engineering so that per-player rolling aggregations use the correct,
#    fully-disambiguated identifier (neither conflating transliteration
#    variants nor mixing homonym careers).
core_frame = assign_player_uuids(core_frame)
core_frame = reconcile_uuids_via_code(core_frame)
core_frame = split_homonym_uuids(core_frame)

# 6. Feature engineering: absence flag (§4.8) and lagged rolling form (§4.9).
#    These now group on the clean Player UUID via Player Name Norm
#    (homonyms carry distinct disambiguated norm values, so groupby is safe).
core_frame = flag_consecutive_absences(core_frame)
core_frame = compute_rolling_features(core_frame)

# 7. Final projection and persistence (§4.11).
historical_baseline = project_and_persist(core_frame, HISTORICAL_BASELINE_PATH)

# 8. Diagnostics for Section 5 (§4.12).
report = validate_dataset(historical_baseline)
log.info("Pipeline complete.")

14:57:43 | INFO    | Season 2020-21: 38 gameweek files found.


14:57:44 | INFO    | Season 2021-22: 38 gameweek files found.
14:57:44 | INFO    | Season 2022-23: 38 gameweek files found.
14:57:45 | INFO    | Season 2023-24: 38 gameweek files found.


C:\Users\SOFI\AppData\Local\Temp\ipykernel_21880\361725704.py:83: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  season_frames.append(pd.concat(per_gw, ignore_index=True))


14:57:45 | INFO    | Season 2024-25: 38 gameweek files found.
14:57:45 | INFO    | Loaded 133,647 rows across 5 seasons.
14:57:46 | INFO    | Season 2020-21: 713 players in metadata table.
14:57:46 | INFO    | Season 2021-22: 737 players in metadata table.
14:57:46 | INFO    | Season 2022-23: 778 players in metadata table.
14:57:46 | INFO    | Season 2023-24: 865 players in metadata table.
14:57:46 | INFO    | Season 2024-25: 804 players in metadata table.
14:57:46 | INFO    | Building core frame for season 2020-21 …
14:57:46 | INFO    | Building core frame for season 2021-22 …
14:57:46 | INFO    | Building core frame for season 2022-23 …
14:57:46 | INFO    | Building core frame for season 2023-24 …
14:57:47 | INFO    | Building core frame for season 2024-25 …
14:57:48 | INFO    | Position repair: filled 342 of 342 missing rows; 0 remain unresolved.
14:57:48 | INFO    | Dropped 13,284 source-CSV duplicate rows (by season: {'2020-21': 3032, '2021-22': 4390, '2022-23': 3113, '2023-24': 1

## 5. Results and Validation

This section reports the headline diagnostics of the produced artefact.
All checks are computed against the in-memory `historical_baseline`
dataframe and the `report` object returned by the orchestration cell.

In [91]:
"""Section 5.1 — Headline summary."""
report.as_summary_table()

,Value
Total rows,133647
Seasons covered,"2020-21, 2021-22, 2022-23, 2023-24, 2024-25"
"Duplicate (UUID, season, GW, Opponent) rows",0
Legitimate DGW rows (Fixture Index > 1),6598
Missing Opponent Difficulty,0
Unresolved Position rows,0
Null Player UUID rows,0


In [92]:
"""Diagnostic — characterise the null-UUID and duplicate populations."""

# Which players are unmapped?
unmapped = (
    historical_baseline[historical_baseline["Player UUID"].isna()]
    .groupby("Player Name", dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
)
print("Top 20 unmapped player names (rows with null UUID):")
print(unmapped)

# How are the duplicates distributed?
dup_mask = historical_baseline.duplicated(
    subset=["Player UUID", "season", "Gameweek"], keep=False
)
dup_with_null = historical_baseline[dup_mask & historical_baseline["Player UUID"].isna()]
dup_real = historical_baseline[dup_mask & historical_baseline["Player UUID"].notna()]

print(f"\nDuplicate rows total : {dup_mask.sum():,}")
print(f"  ... with null UUID : {len(dup_with_null):,}")
print(f"  ... with real UUID : {len(dup_real):,}")

# Confirm which mapping branch was used
print(f"\nCleaned mapping exists? {CLEANED_UUID_MAPPING_PATH.exists()}")
print(f"Auto mapping exists?    {UUID_MAPPING_PATH.exists()}")

Top 20 unmapped player names (rows with null UUID):
Series([], dtype: int64)

Duplicate rows total : 13,157
  ... with null UUID : 0
  ... with real UUID : 13,157

Cleaned mapping exists? True
Auto mapping exists?    True


In [93]:
"""Diagnostic — characterise the duplicate population.

Separates legitimate Double Gameweeks (same player, same GW, DIFFERENT
opponents) from true duplicates (same player, same GW, SAME opponent).
The former must be preserved; only the latter may be deduplicated.
"""

dup_mask = historical_baseline.duplicated(
    subset=["Player UUID", "season", "Gameweek"], keep=False
)
dups = historical_baseline[dup_mask].copy()

# Classify each duplicate group
grp = dups.groupby(["Player UUID", "season", "Gameweek"])

classification = grp.agg(
    n_rows=("Player UUID", "size"),
    n_distinct_opponents=("Opponent Name", "nunique"),
    total_minutes=("Minutes Played", "sum"),
    total_points=("Total Points", "sum"),
).reset_index()

legitimate_dgw = classification[classification["n_distinct_opponents"] == classification["n_rows"]]
true_duplicates = classification[classification["n_distinct_opponents"] < classification["n_rows"]]

print("=" * 70)
print("DUPLICATE CLASSIFICATION")
print("=" * 70)
print(f"Total duplicate-key rows                   : {len(dups):,}")
print(f"  ... classified as legitimate DGW         : {(legitimate_dgw['n_rows']).sum():,}")
print(f"      (distinct opponents per duplicate GW)")
print(f"  ... classified as TRUE duplicates        : {(true_duplicates['n_rows']).sum():,}")
print(f"      (same opponent appearing twice)")
print()
print(f"Distinct (player, season, GW) groups       : {len(classification):,}")
print(f"  ... legitimate DGW groups                : {len(legitimate_dgw):,}")
print(f"  ... true-duplicate groups                : {len(true_duplicates):,}")

print("\n" + "=" * 70)
print("DGW size distribution (matches per GW per player)")
print("=" * 70)
print(legitimate_dgw["n_rows"].value_counts().sort_index())

print("\n" + "=" * 70)
print("DGWs by season (rows affected)")
print("=" * 70)
dgw_rows = dups.merge(
    legitimate_dgw[["Player UUID", "season", "Gameweek"]],
    on=["Player UUID", "season", "Gameweek"], how="inner"
)
print(dgw_rows.groupby("season").size())

print("\n" + "=" * 70)
print("DGWs by (season, Gameweek) — sanity check against real FPL history")
print("=" * 70)
print(dgw_rows.groupby(["season", "Gameweek"]).size().to_string())

if len(true_duplicates) > 0:
    print("\n" + "=" * 70)
    print("SAMPLE of true duplicates (first 10 groups)")
    print("=" * 70)
    sample_keys = true_duplicates.head(10)[["Player UUID", "season", "Gameweek"]]
    sample = dups.merge(sample_keys, on=["Player UUID", "season", "Gameweek"], how="inner")
    display(sample[[
        "Player UUID", "Player Name", "season", "Gameweek",
        "Opponent Name", "Minutes Played", "Total Points"
    ]].sort_values(["Player UUID", "season", "Gameweek"]))
else:
    print("\nNo true duplicates detected — all duplicate-key rows are legitimate DGWs.")

DUPLICATE CLASSIFICATION
Total duplicate-key rows                   : 13,157
  ... classified as legitimate DGW         : 13,157
      (distinct opponents per duplicate GW)
  ... classified as TRUE duplicates        : 0
      (same opponent appearing twice)

Distinct (player, season, GW) groups       : 6,559
  ... legitimate DGW groups                : 6,559
  ... true-duplicate groups                : 0

DGW size distribution (matches per GW per player)
n_rows
2    6520
3      39
Name: count, dtype: int64

DGWs by season (rows affected)
season
2020-21    2913
2021-22    4434
2022-23    3096
2023-24    1966
2024-25     748
dtype: int64

DGWs by (season, Gameweek) — sanity check against real FPL history
season   Gameweek
2020-21  19          692
         24          264
         25          138
         26          946
         27          140
         32           66
         35          667
2021-22  21          136
         22          272
         23          152
         25         

In [94]:
"""Section 5.2 — Records per season."""
records_table = report.rows_per_season.to_frame()
records_table["share_%"] = (records_table["rows"] / records_table["rows"].sum() * 100).round(2)
records_table

,rows,share_%
season,,
2020-21,24365,18.2300
2021-22,25447,19.0400
2022-23,26505,19.8300
2023-24,29725,22.2400
2024-25,27605,20.6600


In [95]:
"""Section 5.3 — Column completeness across seasons (percentage non-null)."""
report.completeness

season,2020-21,2021-22,2022-23,2023-24,2024-25
column,,,,,
Player UUID,100.0000,100.0000,100.0000,100.0000,100.0000
Code,100.0000,100.0000,100.0000,100.0000,100.0000
Player Name,100.0000,100.0000,100.0000,100.0000,100.0000
Web Name,100.0000,100.0000,100.0000,100.0000,100.0000
Player Team ID,100.0000,100.0000,100.0000,100.0000,100.0000
Player Team Name,100.0000,100.0000,100.0000,100.0000,100.0000
season,100.0000,100.0000,100.0000,100.0000,100.0000
Gameweek,100.0000,100.0000,100.0000,100.0000,100.0000
Fixture Index,100.0000,100.0000,100.0000,100.0000,100.0000


In [96]:
"""Section 5.4 — Opponent Difficulty completeness by season.

A non-zero count here would indicate that the fixture-derived team
identification of Section 2.3 failed to resolve some rows; this serves
as a direct integrity check of the methodological core.
"""
od_diag = (
    historical_baseline
    .assign(missing=lambda d: d["Opponent Difficulty"].isna())
    .groupby("season")["missing"]
    .agg(missing_rows="sum", total_rows="size")
    .assign(missing_pct=lambda d: (d["missing_rows"] / d["total_rows"] * 100).round(4))
)
od_diag

,missing_rows,total_rows,missing_pct
season,,,
2020-21,0,24365,0.0000
2021-22,0,25447,0.0000
2022-23,0,26505,0.0000
2023-24,0,29725,0.0000
2024-25,0,27605,0.0000


In [97]:
"""Section 5.5 — Sanity check: top 10 historical scorers.

Aggregated over the five seasons in scope. A defensible dataset will
recover canonical high-scoring players (e.g. Salah, Son, De Bruyne,
Kane); deviations from this expectation indicate identity-resolution
failures.
"""
top_scorers = (
    historical_baseline
    .groupby(["Player UUID", "Player Name", "Position"], dropna=False)["Total Points"]
    .sum()
    .reset_index()
    .sort_values("Total Points", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_scorers

,Player UUID,Player Name,Position,Total Points
0,6d0a6a8b-41b9-4748-b174-eb22cf5b86d6,Mohamed Salah,MID,344
1,c98696d5-6ee3-48cb-8491-7bf3c66aba03,Ollie Watkins,FWD,306
2,8baff127-2097-4ed3-a650-9482f0cb3cfa,Erling Haaland,FWD,272
3,b94be940-2996-467f-be3e-4e004fa9dd3c,Mohamed Salah,MID,265
4,7503f340-ab7a-44a7-a801-76bce028eb1d,Harry Kane,FWD,263
5,2992c30c-f96c-416c-ab65-af16ad6020ce,Heung-Min Son,MID,258
6,c675a8f3-fdec-4081-9004-34cec2bfac97,Bruno Miguel Borges Fernandes,MID,244
7,433afb0b-789f-4e19-abc2-445c1c7d5588,Cole Palmer,MID,244
8,40d9b16b-e916-4e56-86b5-158949066507,Harry Kane,FWD,242
9,f946fd89-4cf0-43af-af8d-bcad4b79fca8,Mohamed Salah,MID,239


In [98]:
"""Section 5.6 — Position distribution.

Fantasy Premier League squads have a fixed positional structure; the
proportion of GK / DEF / MID / FWD rows should be approximately
stable across seasons.
"""
position_dist = (
    historical_baseline
    .groupby(["season", "Position"])
    .size()
    .unstack(fill_value=0)
    .pipe(lambda d: (d.div(d.sum(axis=1), axis=0) * 100).round(2))
)
position_dist

Position,DEF,FWD,GK,MID
season,,,,
2020-21,35.4000,12.7800,11.3600,40.4600
2021-22,33.8700,13.3500,11.4400,41.3400
2022-23,34.6500,11.7400,10.5300,43.0800
2023-24,32.3300,12.9700,11.4800,43.2200
2024-25,33.6000,10.9900,10.7400,44.6700


In [99]:
"""Section 5.7 — Final assertion gate.

Integrity invariants that must hold for the artefact to be considered
usable. The duplicate check is performed on the *correct* composite
key that respects Double Gameweeks: a player may legitimately appear
twice in the same gameweek iff the two rows have distinct opponents
and distinct ``Fixture Index`` values.
"""
assert report.duplicate_key_rows == 0, (
    f"Found {report.duplicate_key_rows} rows duplicated on "
    f"(Player UUID, season, Gameweek, Opponent ID). DGWs should "
    f"already be disambiguated by Opponent ID."
)
assert report.null_player_uuid == 0, (
    f"Found {report.null_player_uuid} rows with a null Player UUID."
)
assert set(report.seasons) == set(HISTORICAL_SEASONS), (
    f"Seasons in artefact ({report.seasons}) differ from configured "
    f"scope ({list(HISTORICAL_SEASONS)})."
)

# Additional DGW invariant: within each (UUID, season, GW) group,
# Fixture Index values must form a contiguous 1..N sequence.
fx_check = (
    historical_baseline.groupby(["Player UUID", "season", "Gameweek"])["Fixture Index"]
    .agg(lambda s: list(sorted(s)) == list(range(1, len(s) + 1)))
)
assert fx_check.all(), (
    f"Found {(~fx_check).sum()} (player, season, GW) groups whose "
    f"Fixture Index values are not a contiguous 1..N sequence."
)

log.info("All integrity assertions passed.")

14:59:09 | INFO    | All integrity assertions passed.


## 6. Conclusion and Downstream Use

### 6.1 Artefact Summary

The notebook produces a single CSV at
`data/processed/historical_baseline.csv`. Each row is uniquely keyed
by the tuple `(Player UUID, season, Gameweek)` and carries the canonical
columns specified in Section 4.11, augmented with leakage-safe rolling
form features `Avg_<metric>_L{3,5}` for the metrics in $\mathcal{M}$.

### 6.2 Known Limitations

- **Pre-2020-21 seasons are excluded.** The public mirror's earlier
  files lack ``expected_goals`` and ``expected_assists``, and use a
  different team-encoding for relegated/promoted clubs. Including them
  is feasible but requires additional schema-harmonisation work
  beyond the scope of this notebook.
- **No team-relative features computed here.** Gameweek-share,
  cumulative causal contribution and intra-team contribution rank are
  intentionally deferred to ``live_ingestion_and_merge.ipynb``, which
  has access to both historical and current-season rows simultaneously.
- **UUID resolution is name-based.** Distinct players sharing
  identical normalised names would be conflated. A manual override
  file (``player_uuid_mapping_cleaned.csv``) is supported for such
  cases; no overrides were required in the seasons under study.

### 6.3 Downstream Handoff

The artefact is consumed by:

- **``live_ingestion_and_merge.ipynb``** — appends the in-progress
  current season fetched from the FPL API, applies smart
  deduplication, computes team-relative aggregations, and produces
  ``data/processed/master_training_set.csv``.
- **``forecasting_and_backtest.ipynb``** — performs feature selection,
  trains the points-prediction model under a time-series
  cross-validation scheme, runs the auto-substitution and captaincy
  backtest, and reports out-of-sample RMSE/MAE.

### 6.4 Reproducibility

Re-running this notebook from a clean state with the same input
``data/`` directory will produce a byte-identical artefact, with the
single exception of newly-introduced ``Player UUID`` values when no
prior mapping file exists. Once ``output/player_uuid_mapping.csv`` has
been written by a first run, all subsequent runs are fully
deterministic.

## References

[1] A. Vaastav, "Fantasy Premier League Historical Data," GitHub
    repository, 2024. [Online]. Available:
    https://github.com/vaastav/Fantasy-Premier-League